# Model Deployment - Set Champion Alias

This notebook deploys a model by setting the 'champion' alias in Unity Catalog.

**Purpose:**
- Set the 'champion' alias to the approved model version
- Enable batch inference to use the champion model
- Track deployment history

**Note:** 
- This notebook should only be run in a Databricks Job, as part of MLflow 3.0 Deployment Jobs.
- This does NOT deploy to model serving - it only changes the alias for batch inference.

## Setup

In [0]:
# Define widgets for job parameters
dbutils.widgets.text("model_name", "")
dbutils.widgets.text("model_version", "")
dbutils.widgets.text("alias", "champion")  # Default to 'champion'

In [0]:
# Get parameters
model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
alias = dbutils.widgets.get("alias")

print(f"Deploying Model: {model_name}")
print(f"Version: {model_version}")
print(f"Alias: {alias}")

## Check Current Champion

In [0]:
from mlflow import MlflowClient

# Initialize MLflow client for Unity Catalog
client = MlflowClient(registry_uri="databricks-uc")

# Check if there's already a champion
try:
    current_champion = client.get_model_version_by_alias(model_name, alias)
    print(f"\nCurrent {alias} version: {current_champion.version}")
    print(f"Current {alias} status: {current_champion.status}")
except Exception as e:
    print(f"\nNo existing {alias} alias found (this is OK for first deployment)")
    current_champion = None

## Set Champion Alias

In [0]:
# Get the model version details
model_version_details = client.get_model_version(model_name, model_version)

print(f"\nModel Version Details:")
print(f"  Name: {model_version_details.name}")
print(f"  Version: {model_version_details.version}")
print(f"  Status: {model_version_details.status}")
print(f"  Run ID: {model_version_details.run_id}")

In [0]:
# Set the alias to the new model version
print(f"\nSetting '{alias}' alias to version {model_version}...")

client.set_registered_model_alias(
    name=model_name,
    alias=alias,
    version=model_version
)

print(f"✓ Successfully set '{alias}' alias to version {model_version}")

## Verify Deployment

In [0]:
# Verify the alias was set correctly
verify_champion = client.get_model_version_by_alias(model_name, alias)

print(f"\nVerification:")
print(f"  Model: {model_name}")
print(f"  Alias: {alias}")
print(f"  Version: {verify_champion.version}")
print(f"  Status: {verify_champion.status}")

# Confirm it matches what we set
assert verify_champion.version == model_version, f"Alias not set correctly! Expected {model_version}, got {verify_champion.version}"
print(f"\n✓ Deployment verified successfully!")

## Add Deployment Tags

In [0]:
from datetime import datetime

# Add deployment timestamp tag
deployment_time = datetime.utcnow().isoformat()
client.set_model_version_tag(
    name=model_name,
    version=model_version,
    key="deployment_timestamp",
    value=deployment_time
)

# Add deployment method tag
client.set_model_version_tag(
    name=model_name,
    version=model_version,
    key="deployment_method",
    value="alias_update"
)

# Add alias tag
client.set_model_version_tag(
    name=model_name,
    version=model_version,
    key="deployed_alias",
    value=alias
)

print(f"\n✓ Deployment tags added:")
print(f"  - deployment_timestamp: {deployment_time}")
print(f"  - deployment_method: alias_update")
print(f"  - deployed_alias: {alias}")

## Summary

In [0]:
print("\n" + "="*60)
print("DEPLOYMENT COMPLETE")
print("="*60)
print(f"Model: {model_name}")
print(f"Version: {model_version}")
print(f"Alias: {alias}")
print(f"Timestamp: {deployment_time}")
print("="*60)
print(f"\n✓ Model version {model_version} is now the {alias}!")
print(f"\nBatch inference jobs will now use this model version.")
print(f"Model URI: models:/{model_name}@{alias}")